[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nehirvardar/Hi-cGAN/blob/rna-update/training_pipeline.ipynb)

# Hi-cGAN Eğitim, Tahmin (Prediction) ve Değerlendirme Pipeline Notebook'u

Bu notebook, Hi-cGAN modelini terminal kullanmadan doğrudan Python üzerinden yapılandırmanızı, eğitmenizi, eğitilmiş modellerle Hi-C temas matrisi tahmini (`hicPredict`) yapmanızı ve tahmin edilen temas haritalarını gerçek Hi-C verisi ile kromozom bazında karşılaştırıp görselleştirmenizi (heatmap, split-diagonal, mesafe bozunumu eğrisi) sağlar.

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab ortamı algılandı!")
    REPO_DIR = '/content/Hi-cGAN'
    if not os.path.exists(REPO_DIR):
        print("GitHub reposu ve veri setleri indiriliyor...")
        # rna-update branch'i veya varsayılan branch'i klonla
        !git clone -b rna-update https://github.com/nehirvardar/Hi-cGAN.git {REPO_DIR} || !git clone https://github.com/nehirvardar/Hi-cGAN.git {REPO_DIR}
    
    %cd {REPO_DIR}
    
    print("Gerekli kütüphaneler kuruluyor")
    !pip install -q cooler pybigwig optuna icecream "ray[tune]" hyperopt
    !pip install -q -e .
    print("\n✅ Colab kurulumu tamamlandı! Çalışma dizini:", os.getcwd())
else:
    print("💻 Yerel geliştirme ortamı aktif:", os.getcwd())


In [1]:
import sys
import os
from datetime import datetime
import tensorflow as tf
from hicgan.training import main as run_hic_training
from hicgan.predict import main as run_hic_prediction
from hicgan.lib import utils

# Proje kök dizininde olduğumuzu doğrulayalım ve GPU durumunu kontrol edelim
print("Mevcut dizin:", os.getcwd())
print("Kullanılabilir GPU sayısı:", len(tf.config.list_physical_devices('GPU')))

Mevcut dizin: /Users/erenyemsen/Documents/Projects/PycharmProjects/Hi-cGAN
Kullanılabilir GPU sayısı: 1


### 1. Genel Pipeline Konfigürasyonu (CONFIG)

Tüm eğitim, tahmin, değerlendirme ve dosya yolu parametrelerini aşağıdaki merkezi `CONFIG` sözlüğü üzerinden kolayca ayarlayabilirsiniz.

In [2]:
# ==============================================================================
# ⚙️ GENEL Hİ-CGAN PİPELİNE KONFİGÜRASYONU
# Tüm eğitim, tahmin ve değerlendirme parametrelerini buradan ayarlayabilirsiniz.
# ==============================================================================

CONFIG = {
    # --- 1. Veri ve Çözünürlük Ayarları ---
    "BIN_SIZE": 250000,           # Hi-C çözünürlüğü (250 kb = 250000)
    "WINDOW_SIZE": 32,           # Model pencere boyutu (250kb için 32 [8 Mb] veya 16 [4 Mb] önerilir)
    "FLANKING_SIZE": 0,          # 2D NPZ girdisi için flanking boyutu (0 = telomer kaybı yok)
    "BATCH_SIZE": 8,              # Eğitim ve tahmin batch boyutu
    
    # --- 2. Veri Yolları (Data Paths) ---
    "NPZ_DIR": "./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count",
    "MEASURED_HIC_PATH": "./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool", # Tüm genom ölçülmüş Hi-C (.cool) matrisi
    "OUTPUT_MODELS_BASE": "./output_models",
    
    # --- 3. Model Eğitimi (Training) Ayarları ---
    "TRAIN_EPOCHS": 10,           # Eğitilecek epoch sayısı
    "TRAIN_CHROMS": ['1', '3', '5', '7', '9',
                     '11', '13', '15', '17', '21'], # Eğitimde kullanılacak kromozom(lar) (Orijinal Hi-cGAN dağılımı)
    "VAL_CHROMS": ["19"],          # Doğrulamada kullanılacak kromozom(lar) (Orijinal Hi-cGAN dağılımı)
    "RUN_NAME": None,             # Özel isim verilebilir (None ise run_YYYYMMDD_HHMMSS)
    "SEED": 42,                   # Rastgelelik tohumu
    "LEARNING_RATE_GEN": 0.0001,  # Generator öğrenme oranı (varsayılan: 2e-5)
    "LEARNING_RATE_DISC": 0.0002, # Discriminator öğrenme oranı (varsayılan: 1e-6)
    
    # --- 4. Model Tahmini (Prediction) Ayarları ---
    # Seçenekler: "all" (tüm kromozomlar), ["1", "2"] veya "1,2", ya da tek kromozom "2"
    "PREDICT_CHROMS": "1,2",      
    "PREDICT_MODEL_PATH": None,   # None ise son eğitilen model otomatik seçilir
    "PREDICT_MATRIX_NAME": None,  # None ise otomatik adlandırılır (tekil vs birleşik)
    
    # --- 5. Değerlendirme & Görselleştirme (Evaluation) Ayarları ---
    "EVAL_MAX_DISTANCE_BINS": 256, # Metriklerin hesaplanacağı maksimum bant genişliği (diyagonal mesafe)
    "EVAL_COLORMAP": "YlOrRd",    # Isı haritası renk paleti
    "SAVE_FIGURES": True,         # Her kromozom için ayrı comparison_chr*.png kaydet
    "SHOW_FIGURES": True,         # Grafikleri notebook içinde sırayla göster
}

print("✅ Konfigürasyon yüklendi:")
for k, v in CONFIG.items():
    print(f"  • {k:22s}: {v}")


✅ Konfigürasyon yüklendi:
  • BIN_SIZE              : 250000
  • WINDOW_SIZE           : 32
  • FLANKING_SIZE         : 0
  • BATCH_SIZE            : 8
  • NPZ_DIR               : ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count
  • MEASURED_HIC_PATH     : ./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool
  • OUTPUT_MODELS_BASE    : ./output_models
  • TRAIN_EPOCHS          : 10
  • TRAIN_CHROMS          : ['1', '3', '5', '7', '9', '11', '13', '15', '17', '21']
  • VAL_CHROMS            : ['19']
  • RUN_NAME              : None
  • SEED                  : 42
  • LEARNING_RATE_GEN     : 0.0001
  • LEARNING_RATE_DISC    : 0.0002
  • PREDICT_CHROMS        : 1,2
  • PREDICT_MODEL_PATH    : None
  • PREDICT_MATRIX_NAME   : None
  • EVAL_MAX_DISTANCE_BINS: 256
  • EVAL_COLORMAP         : YlOrRd
  • SAVE_FIGURES          : True
  • SHOW_FIGURES          : True


### 2. Model Eğitimi (Training Pipeline)

Aşağıdaki `start_training()` fonksiyonu, `CONFIG` ayarlarını kullanarak eğitimi başlatır.
- Her yeni çalıştırmada `output_models` altında tarih-saat içeren benzersiz bir alt klasör (örneğin `./output_models/run_20261011_190000`) oluşturulur; böylece eski modellerin üzerine yazılmaz.
- İstenirse `start_training(run_name="ozel_deney")` şeklinde parametre ezilebilir.

In [3]:
def resolve_chrom_npz(chrom, npz_dir_or_file):
    """Kromozom için 2D girdi NPZ dosyasını bulur."""
    clean = str(chrom).replace("chr", "")
    if os.path.isfile(npz_dir_or_file):
        return npz_dir_or_file
    if os.path.isdir(npz_dir_or_file):
        for candidate in [f"chr{clean}.npz", f"{clean}.npz"]:
            p = os.path.join(npz_dir_or_file, candidate)
            if os.path.exists(p):
                return p
    raise FileNotFoundError(f"Kromozom {chrom} için {npz_dir_or_file} içinde .npz dosyası bulunamadı!")

def start_training(config=CONFIG, run_name=None, epochs=None, batch_size=None, window_size=None):
    """
    CONFIG sözlüğündeki ayarlara göre Hi-cGAN eğitimini başlatır.
    Orijinal Hi-cGAN CLI modülünün parametrelerini eksiksiz eşler (Tüm Genom Matrisi formatı):
      --trainingMatrices (-tm), --trainingChromatinFolders (-tcp), --trainingChromosomes (-tchroms)
      --validationMatrices (-vm), --validationChromatinFolders (-vcp), --validationChromosomes (-vchroms)
      --windowSize (-ws), --epochs (-ep), --batchSize (-bs), --outputFolder (-o)
      --learningRateGenerator (-lrg), --learningRateDiscriminator (-lrd), --seed
    """
    epochs = epochs if epochs is not None else config["TRAIN_EPOCHS"]
    batch_size = batch_size if batch_size is not None else config["BATCH_SIZE"]
    window_size = window_size if window_size is not None else config["WINDOW_SIZE"]
    run_name = run_name if run_name is not None else config["RUN_NAME"]
    base_output_dir = config["OUTPUT_MODELS_BASE"]
    
    if run_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_name = f"run_{timestamp}"
    
    run_output_dir = os.path.join(base_output_dir, run_name)
    os.makedirs(run_output_dir, exist_ok=True)
    
    # TFRecord ve ara dosyaların izolasyonu için çalışma dizini çevre değişkenini ayarla
    os.environ["HICGAN_RUN_DIR"] = run_output_dir
    
    train_chroms = [str(c).replace("chr", "") for c in config["TRAIN_CHROMS"]]
    val_chroms = [str(c).replace("chr", "") for c in config["VAL_CHROMS"]]
    
    # Orijinal Hi-cGAN formatı: Tek parça tüm genom .cool matrisi ve girdi NPZ klasörü
    measured_hic_matrix = config.get("MEASURED_HIC_PATH", config.get("MEASURED_HIC_DIR"))
    npz_dir = config["NPZ_DIR"]
    
    sys_args = [
        'hicTraining',
        # --- Eğitim Verileri (Tüm Genom Matrisi ve NPZ Klasörü) ---
        '-tm', str(measured_hic_matrix),
        '-tcp', str(npz_dir),
        '-tchroms', *train_chroms,
        # --- Doğrulama (Validation) Verileri ---
        '-vm', str(measured_hic_matrix),
        '-vcp', str(npz_dir),
        '-vchroms', *val_chroms,
        # --- Model ve Eğitim Hiperparametreleri ---
        '-ws', str(window_size),
        '-fls', str(config.get('FLANKING_SIZE', 0)),
        '-ep', str(epochs),
        '-bs', str(batch_size),
        '-o', run_output_dir
    ]
    
    # Opsiyonel öğrenme oranları ve tohum
    if config.get("LEARNING_RATE_GEN") is not None:
        sys_args.extend(['-lrg', str(config["LEARNING_RATE_GEN"])])
    if config.get("LEARNING_RATE_DISC") is not None:
        sys_args.extend(['-lrd', str(config["LEARNING_RATE_DISC"])])
    if config.get("SEED") is not None:
        sys_args.extend(['--seed', str(config["SEED"])])
        
    sys.argv = sys_args
    
    print(f"Eğitim başlatılıyor...")
    print(f"Çıktı Dizini: {run_output_dir}")
    print(f"Eğitim Kromozomları: {train_chroms} | Hedef Matris: {measured_hic_matrix} | Girdi Klasörü: {npz_dir}")
    print(f"Doğrulama Kromozomları: {val_chroms} | Hedef Matris: {measured_hic_matrix} | Girdi Klasörü: {npz_dir}")
    print(f"Epoch: {epochs} | Batch Size: {batch_size} | Window Size: {window_size}")
    
    run_hic_training()
    
    print(f"Eğitim tamamlandı! Modeller kaydedildi: {run_output_dir}")
    return run_output_dir


In [4]:
# Eğitimi başlat ve modellerin kaydedildiği klasörü değişkende sakla
# (Not: Halihazırda eğitilmiş bir modeliniz varsa doğrudan bir sonraki Prediction adımına geçebilirsiniz)
run_dir = start_training()


INFO:hicgan.training:Using single GPU training
INFO:hicgan.training:Physically selected GPU: /physical_device:GPU:0


Eğitim başlatılıyor...
Çıktı Dizini: ./output_models/run_20261012_152918
Eğitim Kromozomları: ['1', '3', '5', '7', '9', '11', '13', '15', '17', '21'] | Hedef Matris: ./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool | Girdi Klasörü: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count
Doğrulama Kromozomları: ['19'] | Hedef Matris: ./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool | Girdi Klasörü: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count
Epoch: 10 | Batch Size: 8 | Window Size: 32
random seed set to 42


DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:pydot:pydot initializing
DEBUG:pydot:pydot 4.0.1
DEBUG:pydot.core:pydot core module initializing
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:pydot:pydot initializing
DEBUG:pydot:pydot 4.0.1
DEBUG:pydot.core:pydot core module initializing
DEBUG:h5py._conv:Creating converter from 3 to 5
DEBUG:hicgan.lib.dataContainer:Attempting to load ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr1.npz as sparse matrix...
DEBUG:hicgan.lib.dataContainer:Successfully loaded sparse matrix from ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr1.npz.
DEBUG:hicgan.lib.dataContainer:Scaling features...
INFO:hicgan.lib.dataContainer:Loaded npz

Loaded cooler matrix ./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool
chr. 1, matshape 996*996 -- min. 0 -- max. 0 -- nnz. 850459
Number of samples: 965 | 965 | 0
Number of samples: 965 | 965 | 0


DEBUG:hicgan.lib.dataContainer:Attempting to load ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr11.npz as sparse matrix...
DEBUG:hicgan.lib.dataContainer:Successfully loaded sparse matrix from ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr11.npz.
DEBUG:hicgan.lib.dataContainer:Scaling features...
INFO:hicgan.lib.dataContainer:Loaded npz matrix from ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr11.npz, shape: (541, 541)
DEBUG:hicgan.lib.dataContainer:Factors: True -- Matrix: True -- windowSize: True -- flankingSize: True -- maximumDistance: True
DEBUG:hicgan.lib.dataContainer:Chromatin folder: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count -- Nr. factors: 1
DEBUG:hicgan.lib.dataContainer:Chromatin folder: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count -- Nr. factors: 1
DEBUG:hicgan.lib.dataContainer:self.nr_factors: 1 -- container.nr_factors: 1
DEBUG:hicgan.lib.dataContainer:self.factorNames: ['chr1.npz'] -- container.factorNames:

...

Number of samples: 523 | 523 | 0
Number of samples: 523 | 523 | 0
Loaded cooler matrix ./01_core/02_inputs/measured_hic/gm12878/gm12878_250kb/gm12878_250kb_chrall.cool
chr. 19, matshape 235*235 -- min. 0 -- max. 0 -- nnz. 50914
Number of samples: 204 | 204 | 0
Number of samples: 204 | 204 | 0
ALL TF RECORDS CREATED!
[['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_1_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_11_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_13_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_15_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_17_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_21_001.tfrecord'], ['./

INFO:hicgan.training:Starting training at 2026-10-12 15:29:28.617622


skipping the architecture diagram ./output_models/run_20261012_152918/generatorModel.png: You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.. Install pydot and graphviz if you want it; training is unaffected.


Epoch 00001: 100%|██████████| 665/665 [01:07<00:00,  9.92it/s, g=3.7077, dt=0.800, df=0.170]  
INFO:absl:Sharding callback duration: 30
Epoch 00002: 100%|██████████| 665/665 [00:59<00:00, 11.09it/s, g=2.5110, dt=0.438, df=0.608, v=3.216]
INFO:absl:Sharding callback duration: 24
Epoch 00003: 100%|██████████| 665/665 [00:58<00:00, 11.29it/s, g=2.1789, dt=0.406, df=0.791, v=3.348]
INFO:absl:Sharding callback duration: 33
Epoch 00004: 100%|██████████| 665/665 [00:59<00:00, 11.09it/s, g=2.4552, dt=0.242, df=0.945, v=3.665]
INFO:absl:Sharding callback duration: 30
Epoch 00005: 100%|██████████| 665/665 [00:59<00:00, 11.23it/s, g=3.5943, dt=0.910, df=0.251, v=3.579]
INFO:absl:Sharding callback duration: 189
Epoch 00006: 100%|██████████| 665/665 [00:59<00:00, 11.12it/s, g=2.4614, dt=0.391, df=0.503, v=2.861]
INFO:absl:Sharding callback duration: 73
Epoch 00007: 100%|██████████| 665/665 [00:59<00:00, 11.25it/s, g=3.6630, dt=0.629, df=0.218, v=3.375]
INFO:absl:Sharding callback duration: 27
Epoch

[['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_1_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_11_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_13_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_15_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_17_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_21_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_3_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_5_001.tfrecord'], ['./output_models/run_20261012_152918/._01_core_chrom_npz_homo_sapiens_GRCh38_250kb_tss_count_7_00

### 3. Prediction (Tahmin) Pipeline

Eğitilmiş Hi-cGAN Generator modelini (`generator_*.keras`) kullanarak 2D TSS NPZ girdisinden Hi-C temas matrisi (`.cool`) tahmini üretir.
- `start_prediction()` çağrıldığında `CONFIG` parametrelerini kullanır.
- `chroms="all"` verilirse veri klasöründeki tüm kromozomlar (`chr1`–`chr22`, `chrX`, `chrY`) otomatik tespit edilip sırayla tahmin edilir.
- `CONFIG["SAVE_SEPARATE_COOLERS"] = True` ise birleşik `.cool` dosyasının yanı sıra her kromozom için bağımsız `predicted_chr{c}_250kb.cool` dosyası da kaydedilir.

In [5]:
def get_available_chromosomes(chrom_folder):
    """NPZ klasöründeki mevcut tüm kromozomları doğal genomik sırada listeler."""
    chroms = []
    for f in os.listdir(chrom_folder):
        if f.endswith(".npz"):
            name = f[:-4]
            chrom = name[3:] if name.startswith("chr") else name
            chroms.append(chrom)
    
    def chrom_sort_key(c):
        if c.isdigit():
            return (0, int(c))
        if c.upper() == 'X':
            return (1, 23)
        if c.upper() == 'Y':
            return (1, 24)
        if c.upper() in ['M', 'MT']:
            return (1, 25)
        return (2, c)
        
    return sorted(chroms, key=chrom_sort_key)

def start_prediction(config=CONFIG, chroms=None, model_path=None, matrix_name=None):
    """
    Eğitilmiş model ile 2D NPZ verisi üzerinden Hi-C matris tahmini yapar.
    chroms, model_path veya matrix_name verilmezse CONFIG içindeki değerler kullanılır.
    """
    if chroms is None:
        chroms = config["PREDICT_CHROMS"]
    if model_path is None:
        model_path = config["PREDICT_MODEL_PATH"]
        
    # Model yolunu belirle (verilmemişse en son run klasöründeki son modeli bul)
    if model_path is None:
        base_dir = config["OUTPUT_MODELS_BASE"]
        if not os.path.exists(base_dir):
            raise FileNotFoundError(f"{base_dir} klasörü bulunamadı!")
            
        subdirs = sorted([
            os.path.join(base_dir, d) for d in os.listdir(base_dir)
            if os.path.isdir(os.path.join(base_dir, d)) 
            and (d.startswith("run_") or any(f.startswith("generator") and f.endswith(".keras") for f in os.listdir(os.path.join(base_dir, d))))
        ])
        run_dir = subdirs[-1] if subdirs else base_dir
        
        keras_models = sorted([
            f for f in os.listdir(run_dir)
            if f.startswith("generator") and f.endswith(".keras")
        ])
        if not keras_models:
            raise FileNotFoundError(f"{run_dir} içinde generator .keras modeli bulunamadı!")
        model_path = os.path.join(run_dir, keras_models[-1])
        
    pred_output_dir = os.path.join(os.path.dirname(model_path), "predictions")
    os.makedirs(pred_output_dir, exist_ok=True)
    
    # Kromozom listesini çözümle
    npz_folder = config["NPZ_DIR"]
    if str(chroms).lower() == "all":
        chroms_list = get_available_chromosomes(npz_folder)
    elif isinstance(chroms, str):
        chroms_list = [c.strip().replace("chr", "") for c in chroms.split(",") if c.strip()]
    else:
        chroms_list = [str(c).strip().replace("chr", "") for c in chroms]
        
    if not chroms_list:
        raise ValueError("Tahmin yapılacak en az bir kromozom belirtilmelidir!")

    # Çıktı dosya adını belirle
    if matrix_name is None:
        matrix_name = config["PREDICT_MATRIX_NAME"]
    if matrix_name is None:
        if len(chroms_list) == 1:
            matrix_name = f"predicted_chr{chroms_list[0]}_250kb.cool"
        else:
            matrix_name = "predicted_250kb.cool"

    print(f"Kullanılan Model: {model_path}")
    print(f"Tahmin Yapılacak Kromozom(lar) ({len(chroms_list)} adet): {chroms_list}")
    print(f"Tahmin Çıktı Dosyası: {os.path.join(pred_output_dir, matrix_name)}")
    
    sys.argv = [
        'hicPredict',
        '-trm', model_path,
        '-tcp', npz_folder,
        '-pc', *chroms_list,
        '-b', str(config["BIN_SIZE"]),
        '-ws', str(config["WINDOW_SIZE"]),
        '-fls', str(config.get('FLANKING_SIZE', 0)),
        '-bs', str(config["BATCH_SIZE"]),
        '-o', pred_output_dir,
        '-mn', matrix_name
    ]
    
    print("Tahmin (Prediction) modülü çalıştırılıyor...")
    run_hic_prediction()
    
    result_cool = os.path.join(pred_output_dir, matrix_name)

    # İsteğe bağlı: Çoklu kromozomları tekil .cool dosyalarına da kaydet
    if config.get("SAVE_SEPARATE_COOLERS", True) and len(chroms_list) > 1 and os.path.exists(result_cool):
        print("📁 İsteğe bağlı tekil kromozom .cool dosyaları oluşturuluyor...")
        matrix_chunks_dir = os.path.join(pred_output_dir, "matrix_chunks")
        for chrom in chroms_list:
            npz_chunk = os.path.join(matrix_chunks_dir, f"matrix_chr{chrom}.npz")
            single_cool = os.path.join(pred_output_dir, f"predicted_chr{chrom}_250kb.cool")
            if os.path.exists(npz_chunk):
                utils.writeCooler([npz_chunk], config["BIN_SIZE"], single_cool, [chrom])
                print(f"  • Ayrı Cooler kaydedildi: {single_cool}")

    print(f"Tahmin tamamlandı! Üretilen Cooler dosyası: {result_cool}")
    return result_cool


In [6]:
# Tahmini çalıştır (CONFIG["PREDICT_CHROMS"] değerini kullanır; örn. "1,2", "all" veya "2")
pred_cool_path = start_prediction()


INFO:hicgan.predict:Physically selected GPU: /physical_device:GPU:0
INFO:hicgan.predict:Other GPUs are now invisible to TensorFlow.
DEBUG:hicgan.lib.dataContainer:Attempting to load ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr1.npz as sparse matrix...
DEBUG:hicgan.lib.dataContainer:Successfully loaded sparse matrix from ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr1.npz.
DEBUG:hicgan.lib.dataContainer:Scaling features...
INFO:hicgan.lib.dataContainer:Loaded npz matrix from ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count/chr1.npz, shape: (996, 996)
DEBUG:hicgan.lib.dataContainer:Factors: True -- Matrix: True -- windowSize: True -- flankingSize: True -- maximumDistance: True
DEBUG:hicgan.lib.dataContainer:Chromatin folder: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count -- Nr. factors: 1
DEBUG:hicgan.lib.dataContainer:Chromatin folder: ./01_core/chrom_npz_homo_sapiens_GRCh38_250kb_tss_count -- Nr. factors: 1
DEBUG:hicgan.lib.dataContainer:self

Kullanılan Model: ./output_models/run_20261012_152918/generator_00009.keras
Tahmin Yapılacak Kromozom(lar) (2 adet): ['1', '2']
Tahmin Çıktı Dosyası: ./output_models/run_20261012_152918/predictions/predicted_250kb.cool
Tahmin (Prediction) modülü çalıştırılıyor...
Number of samples: 965 | 0 | 0
Number of samples: 965 | 0 | 0
Number of samples: 938 | 0 | 0


DEBUG:hicgan.lib.dataContainer:Prepare dict... DONE!
DEBUG:hicgan.lib.dataContainer:Write TFRecord...
DEBUG:hicgan.lib.dataContainer:Write TFRecord... DONE!


Number of samples: 938 | 0 | 0
Attempting to load model from: ./output_models/run_20261012_152918/generator_00009.keras


INFO:hicgan.lib.utils:rebuilding with 2 processes (~0.0 GB each, 0.0 GB available)
INFO:hicgan.predict:Overlapping rebuild with prediction, 2 process(es)


...

  chr1   rebuilt, 31,376 stored pixels
  chr2   rebuilt, 30,512 stored pixels


INFO:hicgan.predict:Writing predicted matrix to disk on ./output_models/run_20261012_152918/predictions/predicted_250kb.cool...
INFO:hicgan.lib.utils:Processing chromosome 1 (1/2)
INFO:hicgan.lib.utils:Processing chromosome 2 (2/2)
INFO:hicgan.lib.utils:Writing cooler file to ./output_models/run_20261012_152918/predictions/predicted_250kb.cool
DEBUG:hicgan.lib.utils:Generating pixels for matrix with offset 0
DEBUG:hicgan.lib.utils:Number of nonzero upper-triangular entries: 31376
DEBUG:hicgan.lib.utils:Generating pixels for matrix with offset 996
DEBUG:hicgan.lib.utils:Number of nonzero upper-triangular entries: 30512
INFO:hicgan.lib.utils:Processing chromosome 1 (1/1)


📁 İsteğe bağlı tekil kromozom .cool dosyaları oluşturuluyor...


INFO:hicgan.lib.utils:Writing cooler file to ./output_models/run_20261012_152918/predictions/predicted_chr1_250kb.cool
DEBUG:hicgan.lib.utils:Generating pixels for matrix with offset 0
DEBUG:hicgan.lib.utils:Number of nonzero upper-triangular entries: 31376
INFO:hicgan.lib.utils:Processing chromosome 2 (1/1)
INFO:hicgan.lib.utils:Writing cooler file to ./output_models/run_20261012_152918/predictions/predicted_chr2_250kb.cool
DEBUG:hicgan.lib.utils:Generating pixels for matrix with offset 0
DEBUG:hicgan.lib.utils:Number of nonzero upper-triangular entries: 30512


  • Ayrı Cooler kaydedildi: ./output_models/run_20261012_152918/predictions/predicted_chr1_250kb.cool
  • Ayrı Cooler kaydedildi: ./output_models/run_20261012_152918/predictions/predicted_chr2_250kb.cool
Tahmin tamamlandı! Üretilen Cooler dosyası: ./output_models/run_20261012_152918/predictions/predicted_250kb.cool


### 4. Tahmin Edilen Cooler Matrisini İnceleme

Üretilen `.cool` dosyasının geçerliliğini ve içindeki kromozomları kontrol edin.

In [7]:
import cooler

# Üretilen cooler dosyasını doğrula
c = cooler.Cooler(pred_cool_path)
print("--- Tahmin Edilen Hi-C Matris Özeti ---")
print("Dosya Yolu:", pred_cool_path)
print("Kromozom(lar):", c.chromnames)
print("Bin Boyutu (Çözünürlük):", c.binsize)
print("Matris Boyutu:", c.shape)
print("Toplam Sıfır Olmayan Piksel (NNZ):", c.info['nnz'])


--- Tahmin Edilen Hi-C Matris Özeti ---
Dosya Yolu: ./output_models/run_20261012_152918/predictions/predicted_250kb.cool
Kromozom(lar): ['1', '2']
Bin Boyutu (Çözünürlük): 250000
Matris Boyutu: (1965, 1965)
Toplam Sıfır Olmayan Piksel (NNZ): 61888


### 5. Kromozom Bazlı Değerlendirme, Görselleştirme ve Sayısal Metrikler

Aşağıdaki `evaluate_prediction()` fonksiyonu:
- Tahmin edilen Cooler dosyasındaki **her bir kromozomu tamamen bağımsız ve ayrı ayrı** ele alır.
- Her bir kromozom için gerçek ölçülmüş Hi-C verisi ile karşılaştırmalı **4 panelli yüksek çözünürlüklü görsel (`comparison_chr{chrom}.png`)** üretir (Gerçek, Tahmin, Split-Diagonal ve $P(s)$ Mesafe Bozunumu Eğrisi).
- Her bir kromozom için sayısal metrikleri (Pearson linear & log1p, Spearman, MSE, RMSE, MAE, Cosine Similarity) hesaplar ve **`eval_metrics_chr{chrom}.json`** olarak kaydeder.
- Tüm kromozomlar bittiğinde genel **Özet Tabloyu** ekrana basar ve `eval_metrics_summary.json` ile `.csv` formatında kaydeder.

In [8]:
import cooler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

def find_ground_truth_cooler(chrom, real_cool_path):
    """Belirtilen kromozom için referans ölçülmüş tüm genom Hi-C (.cool) matrisini ve kromozom etiketini bulur."""
    clean_chrom = str(chrom).replace("chr", "")
    if os.path.isfile(real_cool_path):
        c = cooler.Cooler(real_cool_path)
        for name in [clean_chrom, f"chr{clean_chrom}"]:
            if name in c.chromnames:
                return real_cool_path, name
    return None, None

def evaluate_prediction(pred_cool_path, config=CONFIG, chroms=None, output_dir=None):
    """
    Tahmin edilen Hi-C Cooler dosyasını gerçek ölçülmüş veriyle kromozom bazında KARŞILAŞTIRIR:
    - Her bir kromozomu TAMAMEN AYRI AYRI değerlendirir.
    - Her bir kromozom için AYRI bir 4 panelli görsel (comparison_chr{chrom}.png) üretir ve kaydeder.
    - Her bir kromozom için AYRI bir metrik JSON'ı (eval_metrics_chr{chrom}.json) kaydeder.
    - En sonda tüm kromozomların metriklerini ve ortalamasını içeren toplu Özet Tablosu sunar.
    """
    if not os.path.exists(pred_cool_path):
        raise FileNotFoundError(f"Tahmin dosyası bulunamadı: {pred_cool_path}")

    c_pred = cooler.Cooler(pred_cool_path)
    available_pred_chroms = c_pred.chromnames
    real_cool_source = config.get("MEASURED_HIC_PATH", config.get("MEASURED_HIC_DIR"))
    max_distance_bins = config["EVAL_MAX_DISTANCE_BINS"]
    cmap = config["EVAL_COLORMAP"]

    # Değerlendirilecek kromozomları belirle (varsayılan: Cooler içindeki TÜM kromozomlar)
    if chroms is None:
        eval_chroms = available_pred_chroms
    elif isinstance(chroms, str):
        eval_chroms = [c.strip().replace("chr", "") for c in chroms.split(",") if c.strip()]
    else:
        eval_chroms = [str(c).strip().replace("chr", "") for c in chroms]

    if output_dir is None:
        output_dir = os.path.dirname(pred_cool_path)
    os.makedirs(output_dir, exist_ok=True)

    all_metrics = []

    print("=" * 75)
    print(f"📊 Hi-cGAN Kromozom Bazlı Değerlendirme Başlatılıyor")
    print(f"📁 Tahmin Dosyası: {pred_cool_path}")
    print(f"🧬 İncelenecek Kromozomlar ({len(eval_chroms)} adet): {eval_chroms}")
    print("=" * 75)

    for chrom in eval_chroms:
        print(f"\n🔬 [Kromozom {chrom}] Değerlendiriliyor...")
        pred_chrom_name = chrom if chrom in available_pred_chroms else f"chr{chrom}"
        if pred_chrom_name not in available_pred_chroms:
            print(f"  ⚠️ Kromozom {chrom} tahmin dosyasında bulunamadı, atlanıyor.")
            continue

        real_file, real_chrom_name = find_ground_truth_cooler(chrom, real_cool_source)
        if real_file is None:
            print(f"  ⚠️ Kromozom {chrom} için gerçek referans Hi-C dosyası bulunamadı ({real_cool_source}), atlanıyor.")
            continue

        c_real = cooler.Cooler(real_file)
        mat_pred = np.nan_to_num(c_pred.matrix(balance=False).fetch(pred_chrom_name))
        mat_real = np.nan_to_num(c_real.matrix(balance=False).fetch(real_chrom_name))

        min_dim = min(mat_pred.shape[0], mat_real.shape[0])
        mat_pred = mat_pred[:min_dim, :min_dim]
        mat_real = mat_real[:min_dim, :min_dim]

        # Üst üçgen pikseller
        r_idx, c_idx = np.triu_indices(min_dim, k=1)
        dist = c_idx - r_idx
        mask = dist <= max_distance_bins

        y_true = mat_real[r_idx[mask], c_idx[mask]]
        y_pred = mat_pred[r_idx[mask], c_idx[mask]]

        log_true = np.log1p(y_true)
        log_pred = np.log1p(y_pred)

        p_corr, _ = pearsonr(y_true, y_pred) if np.std(y_pred) > 0 else (0.0, 1.0)
        p_corr_log, _ = pearsonr(log_true, log_pred) if np.std(log_pred) > 0 else (0.0, 1.0)
        s_corr, _ = spearmanr(y_true, y_pred) if np.std(y_pred) > 0 else (0.0, 1.0)
        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = float(mean_absolute_error(y_true, y_pred))
        norm_prod = (np.linalg.norm(y_true) * np.linalg.norm(y_pred)) + 1e-9
        cosine_sim = float(np.dot(y_true, y_pred) / norm_prod)

        m = {
            "chromosome": f"chr{chrom}",
            "matrix_shape": f"{min_dim}x{min_dim}",
            "evaluated_pixels": int(mask.sum()),
            "pearson_linear": float(p_corr),
            "pearson_log1p": float(p_corr_log),
            "spearman": float(s_corr),
            "cosine_similarity": float(cosine_sim),
            "mae": float(mae),
            "rmse": float(rmse),
            "mse": float(mse)
        }
        all_metrics.append(m)

        # 1. Kromozoma Özel JSON Kaydı
        metric_file = os.path.join(output_dir, f"eval_metrics_chr{chrom}.json")
        with open(metric_file, "w", encoding="utf-8") as f:
            json.dump(m, f, indent=2)

        # 2. Kromozoma Özel 4 Panelli Görselleştirme
        fig, axes = plt.subplots(1, 4, figsize=(24, 5.5))
        vmax_val = np.percentile(log_true, 99) if len(log_true) > 0 else 1.0

        # Panel 1: Gerçek
        im0 = axes[0].imshow(np.log1p(mat_real), cmap=cmap, vmax=vmax_val)
        axes[0].set_title(f"Gerçek (Measured) - chr{chrom}", fontsize=12, fontweight='bold')
        axes[0].set_xlabel("Genomik Bin (250 kb)")
        axes[0].set_ylabel("Genomik Bin (250 kb)")
        plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

        # Panel 2: Model Tahmini
        im1 = axes[1].imshow(np.log1p(mat_pred), cmap=cmap, vmax=vmax_val)
        axes[1].set_title(f"Hi-cGAN Tahmini - chr{chrom}", fontsize=12, fontweight='bold')
        axes[1].set_xlabel("Genomik Bin (250 kb)")
        axes[1].set_ylabel("Genomik Bin (250 kb)")
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

        # Panel 3: Split-Diagonal Karşılaştırma
        split_mat = np.triu(np.log1p(mat_real)) + np.tril(np.log1p(mat_pred), -1)
        im2 = axes[2].imshow(split_mat, cmap=cmap, vmax=vmax_val)
        axes[2].set_title(f"Split Matris (Üst: Gerçek / Alt: Tahmin)", fontsize=12, fontweight='bold')
        axes[2].set_xlabel("Genomik Bin (250 kb)")
        axes[2].set_ylabel("Genomik Bin (250 kb)")
        plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

        # Panel 4: P(s) Mesafe Bozunumu Eğrisi
        max_d = min(min_dim - 1, max_distance_bins)
        d_range = np.arange(1, max_d)
        decay_real = [np.diagonal(mat_real, d).mean() for d in d_range]
        decay_pred = [np.diagonal(mat_pred, d).mean() for d in d_range]
        dist_mb = (d_range * config["BIN_SIZE"]) / 1e6

        axes[3].plot(dist_mb, decay_real, label="Gerçek (Measured)", color="#d95f02", lw=2)
        axes[3].plot(dist_mb, decay_pred, label="Hi-cGAN (Tahmin)", color="#1b9e77", lw=2, linestyle="--")
        axes[3].set_title(f"$P(s)$ Mesafe Bozunumu - chr{chrom}", fontsize=12, fontweight='bold')
        axes[3].set_xlabel("Genomik Mesafe (Mb)")
        axes[3].set_ylabel("Ortalama Temas Yoğunluğu")
        axes[3].set_yscale("log")
        axes[3].legend(loc="upper right")
        axes[3].grid(True, which="both", ls=":", alpha=0.6)

        plt.tight_layout()
        plot_path = os.path.join(output_dir, f"comparison_chr{chrom}.png")
        if config["SAVE_FIGURES"]:
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
            print(f"  🖼️ Görsel Kaydedildi: {plot_path}")
        if config["SHOW_FIGURES"]:
            plt.show()
        else:
            plt.close(fig)

        print(f"  📈 Metrikler: Pearson(Log1p): {m['pearson_log1p']:.4f} | Spearman: {m['spearman']:.4f} | MAE: {m['mae']:.4f}")

    # 3. Genel Özet Tablosu
    if all_metrics:
        df = pd.DataFrame(all_metrics)
        numeric_cols = ["pearson_linear", "pearson_log1p", "spearman", "cosine_similarity", "mae", "rmse"]
        avg_row = {col: df[col].mean() for col in numeric_cols}
        avg_row["chromosome"] = "ORTALAMA (MEAN)"
        avg_row["matrix_shape"] = "-"
        avg_row["evaluated_pixels"] = int(df["evaluated_pixels"].sum())
        avg_row["mse"] = df["mse"].mean()
        df_summary = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

        print("\n" + "=" * 90)
        print("📈 GENEL DEĞERLENDİRME METRİKLERİ ÖZET TABLOSU")
        print("=" * 90)
        print(df_summary[["chromosome", "pearson_linear", "pearson_log1p", "spearman", "cosine_similarity", "mae", "rmse"]].to_string(index=False))
        print("=" * 90)

        summary_json = os.path.join(output_dir, "eval_metrics_summary.json")
        summary_csv = os.path.join(output_dir, "eval_metrics_summary.csv")
        df_summary.to_json(summary_json, indent=2, orient="records")
        df_summary.to_csv(summary_csv, index=False)
        print(f"📄 Toplu Özet Tabloları Kaydedildi: {summary_json} ve {summary_csv}")
        return df_summary
    else:
        print("⚠️ Değerlendirilebilecek hiçbir kromozom eşleşmedi.")
        return None


In [9]:
# Tahmin edilen Cooler dosyasındaki her kromozomu ayrı ayrı değerlendir, görselleştir ve özet tabloyu üret
metrics_df = evaluate_prediction(pred_cool_path)


📊 Hi-cGAN Kromozom Bazlı Değerlendirme Başlatılıyor
📁 Tahmin Dosyası: ./output_models/run_20261012_152918/predictions/predicted_250kb.cool
🧬 İncelenecek Kromozomlar (2 adet): ['1', '2']

🔬 [Kromozom 1] Değerlendiriliyor...
  🖼️ Görsel Kaydedildi: ./output_models/run_20261012_152918/predictions/comparison_chr1.png
  📈 Metrikler: Pearson(Log1p): 0.3149 | Spearman: 0.4467 | MAE: 623.3756

🔬 [Kromozom 2] Değerlendiriliyor...
  🖼️ Görsel Kaydedildi: ./output_models/run_20261012_152918/predictions/comparison_chr2.png
  📈 Metrikler: Pearson(Log1p): 0.4764 | Spearman: 0.5061 | MAE: 652.4341

📈 GENEL DEĞERLENDİRME METRİKLERİ ÖZET TABLOSU
     chromosome  pearson_linear  pearson_log1p  spearman  cosine_similarity        mae        rmse
           chr1        0.901773       0.314858  0.446738           0.903939 623.375641 3452.815594
           chr2        0.865411       0.476414  0.506113           0.868348 652.434100 3171.152267
ORTALAMA (MEAN)        0.883592       0.395636  0.476426          